# UTS Pemodelan dan Simulasi
Sistem dinamik penularan DBD (manusia SIR, nyamuk SI) dengan data kasus Filipina 2016-2020.
Jalankan sel dari atas ke bawah.

In [ ]:
from google.colab import files
files.upload()  # pilih ph_dengue_cases2016-2020.csv

## 1. Model dan data
Persamaan: dS/dt = -b m(t) Iv S + w R; dI/dt = b m(t) Iv S - g I; dR/dt = g I - w R; dIv/dt = c I (1-Iv) - mu Iv. m(t) = (1 + a cos(2pi(t-t0)/365)) menggambarkan musim hujan.

In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import least_squares
N = 108_000_000          # populasi Filipina (ganti dgn data PSA)
GAMMA, MU, OMEGA = 1/7, 1/14, 1/365   # pemulihan, kematian nyamuk, hilang imun (per hari)
def simulate(p, days=1826, ctrl=0.0, ctrl_start=None, i0=2e-5):
    b, c, a, t0 = p[:4]
    S, I, R, Iv = 1-i0, i0, 0.0, 1e-4
    out = []
    for d in range(days):
        red = ctrl if (ctrl_start is not None and d >= ctrl_start) else 0.0
        m = (1 + a*np.cos(2*np.pi*(d-t0)/365)) * (1-red)
        new = b*m*Iv*S
        dS = -new + OMEGA*R
        dI = new - GAMMA*I
        dR = GAMMA*I - OMEGA*R
        dIv = c*I*(1-Iv) - MU*Iv
        S += dS; I += dI; R += dR; Iv += dIv
        out.append(new)
    out = np.array(out)
    edges = pd.date_range('2016-01-01', periods=days, freq='D')
    return pd.Series(out*N, index=edges).resample('MS').sum()
def load():
    df = pd.read_csv('ph_dengue_cases2016-2020.csv')
    df['date'] = pd.to_datetime(df.Month+' '+df.Year.astype(str), format='%B %Y')
    return df.groupby('date').Dengue_Cases.sum()
def fit(obs):
    tr = obs[:'2018-12-01']
    def res(q):
        p = [q[0], q[1], q[2], q[3]]; rho = q[4]
        sim = simulate(p)[:'2018-12-01']*rho
        return (sim.values-tr.values)/np.sqrt(tr.values+1)
    best=None
    for b0 in (0.3,0.6):
        r = least_squares(res,[b0,0.3,0.4,200,0.5],bounds=([0.05,0.05,0,0,0.01],[3,3,0.95,365,1]))
        if best is None or r.cost<best.cost: best=r
    return best.x


In [ ]:
obs = load()
obs.plot(figsize=(9,3), title='Kasus DBD per bulan (total nasional)');

## 2. Kalibrasi 2016-2018 dan validasi 2019
Kolom Dengue_Deaths tidak dipakai (ada nilai kematian > kasus). Tahun 2020 tidak dipakai kalibrasi karena terdampak COVID-19.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

q = fit(obs)
print('b, c, a, t0, rho =', q.round(4))
sim = simulate(q[:4]) * q[4]

split_date = pd.Timestamp('2019-01-01')

plt.figure(figsize=(10, 4))
plt.plot(obs, label='Data')
plt.plot(sim, '--', label='Model')
plt.axvline(split_date, c='gray')  # or mdates.date2num(split_date)
plt.legend()
plt.show()

for n, (s, e) in {'Latih 2016-2018': ('2016', '2018'), 'Uji 2019': ('2019', '2019')}.items():
    mape = (abs(sim[s:e] - obs[s:e]) / obs[s:e]).mean() * 100
    print(n, 'MAPE %.1f%%' % mape)

## 3. Skenario pengurangan populasi nyamuk (mulai Januari 2019)

In [ ]:
plt.figure(figsize=(10,4))
for c in (0,0.2,0.4):
    s = simulate(q[:4], ctrl=c, ctrl_start=1096)*q[4]
    plt.plot(s['2019':], label=f'Pengurangan {int(c*100)}%  total {int(s["2019"].sum()):,}')
plt.legend(); plt.show()

## 4. Salin parameter ke web
Jalankan sel di bawah, lalu cocokkan hasilnya dengan objek `P` di `index.html`.

In [ ]:
P = dict(b=float(q[0]), c=float(q[1]), a=float(q[2]), t0=float(q[3]), rho=float(q[4]), N=N)
print('P =', P)